# Gold Layer

## Purpose

The Gold layer contains business-level insights and KPIs generated from the Silver layer.

## KPIs Generated

- Delivery Success Rate
- Failure Rate
- Average Delivery Time
- High-Risk Zones
- Drone Performance

## Benefits

- Business reporting
- Operational monitoring
- Decision-making support

In [0]:
from pyspark.sql.functions import *

In [0]:
deliveries = spark.table("silver_deliveries")

logs = spark.table("silver_logs")

drones = spark.table("bronze_drones")

In [0]:
final_df = deliveries.join(
    logs,
    "delivery_id",
    "inner"
)

In [0]:
display(final_df)

delivery_id,drone_id,source,destination,distance_km,start_time,end_time,ingestion_time,delivery_duration,processed_time,log_id,battery_level,gps_signal,status,ingestion_time,failure_flag,processed_time
1,11,Mumbai,Mumbai,29.63,2026-07-11T16:12:24.651Z,2026-07-11T16:57:24.651Z,2026-07-11T16:25:18.411Z,45.0,2026-07-11T16:31:57.920Z,1,52,79,SIGNAL_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
2,10,Bangalore,Mumbai,14.64,2026-07-11T16:12:24.651Z,2026-07-11T17:58:24.651Z,2026-07-11T16:25:18.411Z,106.0,2026-07-11T16:31:57.920Z,2,39,33,SIGNAL_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
3,11,Bangalore,Pune,12.15,2026-07-11T16:12:24.651Z,2026-07-11T17:17:24.651Z,2026-07-11T16:25:18.411Z,65.0,2026-07-11T16:31:57.920Z,3,44,87,SIGNAL_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
4,4,Hyderabad,Delhi,24.07,2026-07-11T16:12:24.651Z,2026-07-11T16:22:24.651Z,2026-07-11T16:25:18.411Z,10.0,2026-07-11T16:31:57.920Z,4,84,61,WEATHER_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
5,4,Mumbai,Hyderabad,2.22,2026-07-11T16:12:24.651Z,2026-07-11T16:51:24.651Z,2026-07-11T16:25:18.411Z,39.0,2026-07-11T16:31:57.920Z,5,78,49,SIGNAL_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
6,14,Hyderabad,Bangalore,6.0,2026-07-11T16:12:24.651Z,2026-07-11T16:25:24.651Z,2026-07-11T16:25:18.411Z,13.0,2026-07-11T16:31:57.920Z,6,72,80,BATTERY_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
7,10,Mumbai,Hyderabad,22.28,2026-07-11T16:12:24.651Z,2026-07-11T16:27:24.651Z,2026-07-11T16:25:18.411Z,15.0,2026-07-11T16:31:57.920Z,7,14,58,WEATHER_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
8,15,Hyderabad,Mumbai,12.37,2026-07-11T16:12:24.651Z,2026-07-11T16:43:24.651Z,2026-07-11T16:25:18.411Z,31.0,2026-07-11T16:31:57.920Z,8,94,61,WEATHER_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
9,7,Mumbai,Mumbai,38.03,2026-07-11T16:12:24.651Z,2026-07-11T16:57:24.651Z,2026-07-11T16:25:18.411Z,45.0,2026-07-11T16:31:57.920Z,9,62,36,BATTERY_FAILURE,2026-07-11T16:25:22.276Z,1,2026-07-11T16:32:01.375Z
10,1,Mumbai,Mumbai,37.71,2026-07-11T16:12:24.651Z,2026-07-11T16:35:24.651Z,2026-07-11T16:25:18.411Z,23.0,2026-07-11T16:31:57.920Z,10,13,83,SUCCESS,2026-07-11T16:25:22.276Z,0,2026-07-11T16:32:01.375Z


In [0]:
gold_kpis = final_df.groupBy(
    "destination"
).agg(

    count("*").alias(
        "total_deliveries"
    ),

    round(
        avg("delivery_duration"),
        2
    ).alias(
        "avg_delivery_time"
    ),

    round(
        avg("failure_flag") * 100,
        2
    ).alias(
        "failure_rate_percent"
    ),

    round(
        (1 - avg("failure_flag")) * 100,
        2
    ).alias(
        "success_rate_percent"
    )
)

In [0]:
high_risk_zones = final_df.filter(
    col("failure_flag") == 1
).groupBy(
    "destination"
).agg(

    count("*").alias(
        "failure_count"
    )

).orderBy(
    desc("failure_count")
)

In [0]:
gold_kpis.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_kpis")

high_risk_zones.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_high_risk_zones")

In [0]:
display(gold_kpis)

destination,total_deliveries,avg_delivery_time,failure_rate_percent,success_rate_percent
Delhi,36,57.06,61.11,38.89
Hyderabad,42,59.12,76.19,23.81
Pune,40,69.18,75.0,25.0
Bangalore,38,70.71,76.32,23.68
Mumbai,44,57.09,75.0,25.0


In [0]:
display(high_risk_zones)

destination,failure_count
Mumbai,33
Hyderabad,32
Pune,30
Bangalore,29
Delhi,22


Individual drone analysis

In [0]:
drone_performance = final_df.groupBy(
    "drone_id"
).agg(

    count("*").alias(
        "total_deliveries"
    ),

    round(
        avg("failure_flag") * 100,
        2
    ).alias(
        "failure_rate_percent"
    )
)

display(drone_performance)

drone_id,total_deliveries,failure_rate_percent
11,9,66.67
9,7,71.43
14,12,75.0
17,4,75.0
3,9,66.67
4,16,93.75
19,12,66.67
6,9,100.0
13,8,62.5
20,9,22.22


Query 1: View the Gold KPI Table

In [0]:
%sql

SELECT * FROM gold_kpis;

destination,total_deliveries,avg_delivery_time,failure_rate_percent,success_rate_percent
Delhi,36,57.06,61.11,38.89
Hyderabad,42,59.12,76.19,23.81
Pune,40,69.18,75.0,25.0
Bangalore,38,70.71,76.32,23.68
Mumbai,44,57.09,75.0,25.0


Query 2: Average Delivery Time by Destination

In [0]:
%sql

SELECT
    destination,
    avg_delivery_time
FROM gold_kpis
ORDER BY avg_delivery_time DESC;

destination,avg_delivery_time
Bangalore,70.71
Pune,69.18
Hyderabad,59.12
Mumbai,57.09
Delhi,57.06


Databricks visualization. Run in Databricks to view.

Query 3: Success Rate Analysis

In [0]:
%sql

SELECT
    destination,
    success_rate_percent
FROM gold_kpis
ORDER BY success_rate_percent DESC;

destination,success_rate_percent
Delhi,38.89
Pune,25.0
Mumbai,25.0
Hyderabad,23.81
Bangalore,23.68


Databricks visualization. Run in Databricks to view.

Query 4: Failure Rate Analysis

In [0]:
%sql

SELECT
    destination,
    failure_rate_percent
FROM gold_kpis
ORDER BY failure_rate_percent DESC;

destination,failure_rate_percent
Bangalore,76.32
Hyderabad,76.19
Pune,75.0
Mumbai,75.0
Delhi,61.11


Databricks visualization. Run in Databricks to view.

Query 5: High-Risk Zones

In [0]:
%sql

SELECT *
FROM gold_high_risk_zones;

destination,failure_count
Mumbai,33
Hyderabad,32
Pune,30
Bangalore,29
Delhi,22


Databricks visualization. Run in Databricks to view.

Query 6: Top 5 Riskiest Areas

In [0]:
%sql

SELECT
    destination,
    failure_count
FROM gold_high_risk_zones
ORDER BY failure_count DESC
LIMIT 5;

destination,failure_count
Mumbai,33
Hyderabad,32
Pune,30
Bangalore,29
Delhi,22


In [0]:
drone_performance.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_drone_performance")

In [0]:
%sql

SELECT * FROM gold_drone_performance;

drone_id,total_deliveries,failure_rate_percent
11,9,66.67
9,7,71.43
14,12,75.0
17,4,75.0
3,9,66.67
4,16,93.75
19,12,66.67
6,9,100.0
13,8,62.5
20,9,22.22


Databricks visualization. Run in Databricks to view.